# Bài 15: Multi-company Comparison
## So sánh nhiều công ty trong một câu hỏi

**Mục tiêu:**
- Mở rộng state từ `symbol: str` → `symbols: List[str]`
- Thu thập dữ liệu song song cho nhiều công ty
- Dùng LLM tổng hợp và so sánh

---
## Phần 1: Lý thuyết

### Vấn đề hiện tại

Orchestrator ở các bài trước chỉ xử lý **1 công ty mỗi lần**:
```python
state = {"question": "...", "symbol": "NVDA", ...}
```

Nếu user hỏi *"So sánh NVDA và AMD về doanh thu và giá cổ phiếu"* → hệ thống không biết làm.

---

### Giải pháp: `symbols: List[str]` + Data Collection Node

Thay đổi state:
```python
state = {
    "question": "So sánh NVDA và AMD",
    "symbols": ["NVDA", "AMD"],      # ← thay vì symbol: str
    "company_data": {},               # ← {"NVDA": "...", "AMD": "..."}
    "comparison": ""
}
```

Graph đơn giản hơn Bài 14 (không cần router vì ta biết rõ cần gì):
```
START → data_collection → comparison → END
```

- **`data_collection_node`**: loop qua từng symbol, gọi yfinance + RAG, lưu vào `company_data`
- **`comparison_node`**: gom tất cả `company_data`, hỏi LLM viết bài so sánh

---
## Phần 2: Ví dụ — So sánh đơn giản với yfinance

Trước khi tích hợp LangGraph và RAG, xem cách thu thập và so sánh dữ liệu từ nhiều công ty:

In [ ]:
import yfinance as yf

def get_financial_snapshot(symbol: str) -> str:
    """Lấy snapshot tài chính cơ bản từ yfinance."""
    ticker = yf.Ticker(symbol)
    info = ticker.info
    return (
        f"Symbol: {symbol}\n"
        f"  Giá hiện tại : ${info.get('currentPrice', 'N/A')}\n"
        f"  Market Cap   : ${info.get('marketCap', 'N/A'):,}\n"
        f"  P/E ratio    : {info.get('trailingPE', 'N/A')}\n"
        f"  Revenue (TTM): ${info.get('totalRevenue', 'N/A'):,}"
    )


# Thu thập dữ liệu cho nhiều công ty
symbols = ["NVDA", "AMD"]
company_data = {}

for symbol in symbols:
    data = get_financial_snapshot(symbol)
    company_data[symbol] = data
    print(data)
    print()

**Nhận xét:** Pattern cốt lõi rất đơn giản — `for symbol in symbols` rồi thu thập vào dict. Phần bài tập sẽ mở rộng thêm RAG và đưa vào LangGraph.

---
## Phần 3: Bài tập

Build hệ thống so sánh đầy đủ: **yfinance + RAG (ChromaDB từ Bài 13) + LangGraph**.

Việc cần làm:
- ✅ Setup — đã cho sẵn
- ✅ **Bước 1:** Viết `collect_for_symbol(symbol, question)` — gọi cả yfinance lẫn ChromaDB
- ✅ **Bước 2:** Viết `data_collection_node(state)` — loop qua `symbols`, gọi Bước 1
- ✅ `comparison_node` — đã cho sẵn
- ✅ **Bước 3:** Build graph và test

In [ ]:
# ĐÃ CHO SẴN — Setup
import os, json
from typing import TypedDict
from dotenv import load_dotenv
import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from langgraph.graph import StateGraph, START, END

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

CHROMA_PATH = r"D:\Code\AI\Multi-Agent Research Assistant project\phase4\chroma_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection("multi_company_reports")

print(f"ChromaDB: {collection.count()} chunks sẵn sàng")

In [ ]:
# ĐÃ CHO SẴN — State
class ComparisonState(TypedDict):
    question: str
    symbols: list[str]         # ["NVDA", "AMD"]
    company_data: dict         # {"NVDA": "...", "AMD": "..."}
    comparison: str

In [ ]:
# ❓ BƯỚC 1: Viết collect_for_symbol(symbol, question)
#
# Hàm này thu thập thông tin về MỘT công ty từ 2 nguồn:
#   1. yfinance  → giá, market cap, P/E (xem get_financial_snapshot() ở Phần 2)
#   2. ChromaDB  → top 3 chunks từ báo cáo, lọc theo where={"symbol": symbol}
#      (xem lại cách query collection ở Bài 13)
#
# Trả về: 1 string gộp cả 2 nguồn, ví dụ:
#   "=== NVDA ===\n[Tài chính]\n...\n[Báo cáo]\n..."

def collect_for_symbol(symbol: str, question: str) -> str:
    ...

In [ ]:
# ❓ BƯỚC 2: Viết data_collection_node(state)
#
# Gợi ý:
# - Lấy symbols và question từ state
# - Loop qua từng symbol, gọi collect_for_symbol()
# - In [symbol] đang xử lý để theo dõi tiến độ
# - Trả về {"company_data": {symbol: data, ...}}

def data_collection_node(state: ComparisonState):
    ...

In [ ]:
# ĐÃ CHO SẴN — comparison_node
def comparison_node(state: ComparisonState):
    # Gom tất cả company_data thành 1 context lớn
    context = "\n\n".join(state["company_data"].values())

    prompt = (
        f"Dựa vào thông tin sau về các công ty:\n\n{context}\n\n"
        f"Hãy trả lời câu hỏi: {state['question']}\n"
        f"Phân tích có cấu trúc, so sánh trực tiếp giữa các công ty."
    )
    response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    return {"comparison": response.text}

In [ ]:
# ❓ BƯỚC 3: Build graph và test
#
# Graph:
#   START → data_collection → comparison → END
#
# Test với câu hỏi so sánh thật:
#   question = "So sánh NVDA và AMD về doanh thu, lợi nhuận và giá cổ phiếu"
#   symbols  = ["NVDA", "AMD"]

...